# Detector de Vallas Publicitarias

**El caso de uso.** Una plataforma de etiquetado geolocalizado de imágenes: vehículos de relevamiento recorren la
ciudad con una cámara y cada frame llega con su posición GPS (se ve sobreimpresa en las fotos). El sistema es el
**microservicio de detección** pensado para integrarse a esa plataforma: recibe un frame y devuelve dónde hay vallas
publicitarias, para que la plataforma las etiquete en el mapa.

| Pieza | Papel en la integración |
|---|---|
| Gateway REST | la API que la plataforma llama para etiquetar una imagen |
| Redpanda + consumidor | la ingesta de los vehículos: absorbe ráfagas y conserva el orden de cada recorrido |
| `predictions/` en el Data Lake | las etiquetas que consume la plataforma |
| Gate de promoción + rollback | ningún modelo nuevo cambia las etiquetas sin pasar el gate |
| Monitoreo de drift | aviso cuando cambian las condiciones de captura (de noche, otra ciudad, otra cámara) |

Este notebook usa el servicio como lo haría la plataforma. Airflow, MLflow y MinIO se muestran en el navegador.

> Requiere el stack arriba y con champion (`register_model` corrido). Los secretos se leen de `.env` y no se imprimen.

## 0 · Configuración

Las claves salen de `.env`; ninguna celda las muestra.

In [1]:
import io, json, subprocess, sys, time
from datetime import datetime, timezone
from pathlib import Path

import boto3, grpc, requests
import matplotlib.pyplot as plt
import pandas as pd
from matplotlib.patches import Rectangle
from PIL import Image, ImageEnhance

ROOT = Path.cwd().parent
env = dict(line.split("=", 1) for line in (ROOT / ".env").read_text().splitlines()
           if "=" in line and not line.startswith("#"))

GATEWAY = "http://localhost:8800"
HEADERS = {"X-API-Key": env["GATEWAY_API_KEYS"].split(",")[0]}
IMAGES = ROOT / "data/billboards-2025-09-14-sur-lp/images"
IMAGE = IMAGES / "042e2f0b-229c-46b6-8236-1e3391cb3682_1647.jpg"   # from the test split: a video the model never saw

Qué modelo está sirviendo el gateway: versión, alias y las métricas del gate con su intervalo de confianza.

In [2]:
model = requests.get(f"{GATEWAY}/v1/model", headers=HEADERS).json()
print(f"{model['name']} v{model['version']} ({model['alias']}) · threshold {model['score_threshold']}")
print(f"gate set: {model['eval_set']}")
pd.Series(model["metrics"]).round(3)

ConnectionError: HTTPConnectionPool(host='localhost', port=8800): Max retries exceeded with url: /v1/model (Caused by NewConnectionError("HTTPConnection(host='localhost', port=8800): Failed to establish a new connection: [Errno 111] Connection refused"))

## 1 · La plataforma pide etiquetar un frame

`POST /v1/detect` con su API key. El gateway no tiene el modelo: autentica, valida y reenvía la imagen por gRPC al
servicio de inferencia. La respuesta trae las cajas y la **versión del modelo** que las produjo.

In [ ]:
def detect(path, **params):
    with open(path, "rb") as f:
        response = requests.post(f"{GATEWAY}/v1/detect", headers=HEADERS, params=params,
                                 files={"image": (path.name, f, "image/jpeg")})
    response.raise_for_status()
    return response.json()

result = detect(IMAGE)
print(f"model v{result['model']['version']} · {result['latency_ms']:.0f} ms")
pd.DataFrame(result["detections"])

Las mismas cajas dibujadas sobre la foto:

In [ ]:
def draw(path, result, ax):
    ax.imshow(Image.open(path))
    for d in result["detections"]:
        x0, y0, x1, y1 = d["box"]
        ax.add_patch(Rectangle((x0, y0), x1 - x0, y1 - y0, fill=False, edgecolor="#eb6834", linewidth=2))
        ax.text(x0, y0 - 10, f"billboard {d['score']:.2f}", color="white", fontsize=9,
                bbox={"facecolor": "#eb6834", "edgecolor": "none", "pad": 2})
    ax.set_title(f"{len(result['detections'])} billboards · {result['latency_ms']:.0f} ms", loc="left")
    ax.axis("off")

fig, ax = plt.subplots(figsize=(12, 7))
draw(IMAGE, result, ax)

Otros recorridos, otras calles (todos del test revisado):

In [ ]:
frames = ["62450afd-c5c6-4438-9127-4aa4c21754f3_1671.jpg", "042e2f0b-229c-46b6-8236-1e3391cb3682_2927.jpg",
          "91ff0e05-653a-43bf-a09b-c73aa3a37045_164.jpg", "8ed4343d-a1d1-4dc5-a4a0-70cef4d52c38_1755.jpg"]

fig, axes = plt.subplots(2, 2, figsize=(14, 8.5))
for name, ax in zip(frames, axes.flat):
    draw(IMAGES / name, detect(IMAGES / name), ax)
fig.tight_layout()

## 2 · Adentro: gRPC

Entre el gateway y la inferencia el protocolo es gRPC: la imagen viaja como bytes, con contrato tipado
(`proto/detector.proto`). Es un servicio interno: la plataforma solo ve el gateway. Llamándolo directamente, con el token:

In [ ]:
sys.path.insert(0, str(ROOT))
if not (ROOT / "billboard_detector/grpc_gen").exists():
    subprocess.run(["sh", "scripts/gen_proto.sh"], cwd=ROOT, check=True)
from billboard_detector.grpc_gen import detector_pb2 as pb, detector_pb2_grpc as pb_grpc

stub = pb_grpc.DetectorStub(grpc.insecure_channel("localhost:50051"))
token = [("authorization", f"Bearer {env['INFERENCE_TOKEN']}")]

request = pb.DetectRequest(image=IMAGE.read_bytes())
response = stub.Detect(request, metadata=token)
print(response.model, f"latency_ms: {response.latency_ms:.0f}", f"detections: {len(response.detections)}", sep="\n")

Cuánto pesa la misma imagen en cada formato: gRPC y multipart mandan los bytes tal cual; JSON obliga a base64.

In [ ]:
import base64

image_bytes = IMAGE.read_bytes()
sizes = pd.Series({
    "REST multipart": len(image_bytes),
    "gRPC (bytes)": request.ByteSize(),
    "JSON + base64": len(json.dumps({"image": base64.b64encode(image_bytes).decode()})),
}, name="request bytes")
print(f"base64 adds {sizes['JSON + base64'] / sizes['REST multipart'] - 1:+.0%}")
sizes.to_frame()

## 3 · Seguridad

Un servicio que se integra a una plataforma no puede aceptar cualquier cosa. Cada fila es un ataque real contra el sistema
que está corriendo.

In [ ]:
def post_image(data, headers=HEADERS, content_type="image/jpeg", path="/v1/detect"):
    files = {"image": ("frame.jpg", data, content_type)} if data is not None else None
    return requests.post(GATEWAY + path, headers=headers, files=files).status_code

gif = io.BytesIO(); Image.new("RGB", (64, 64), "red").save(gif, "GIF")             # a GIF disguised as JPEG
bomb = io.BytesIO(); Image.new("1", (20000, 20000)).save(bomb, "PNG")              # 20000x20000 in a few KB

def grpc_without_token():
    try:
        stub.GetModel(pb.GetModelRequest())
        return "OK"
    except grpc.RpcError as e:
        return e.code().name

checks = [
    ("valid image with API key",      200, post_image(image_bytes)),
    ("no API key",                    401, post_image(image_bytes, headers={})),
    ("invalid API key",               403, post_image(image_bytes, headers={"X-API-Key": "not-mine"})),
    ("GIF declared as image/jpeg",    415, post_image(gif.getvalue())),
    ("decompression bomb (PNG)",      422, post_image(bomb.getvalue(), content_type="image/png")),
    ("file larger than 5 MB",         413, post_image(b"0" * 6_000_000)),
    ("reload with a user key",        403, post_image(None, path="/v1/reload")),
    ("gRPC straight to the model, no token", "UNAUTHENTICATED", grpc_without_token()),
]
pd.DataFrame(checks, columns=["attack", "expected", "got"]).assign(
    ok=lambda df: (df.expected == df.got).map({True: "✔", False: "✘"}))

## 4 · Ingesta de los vehículos de relevamiento (Redpanda)

Tres recorridos (videos del test) publican sus frames en el topic `frames`, como lo harían los vehículos. El recorrido es
la clave del mensaje: sus frames caen siempre en la misma partición y quedan en orden. El instante de captura viaja en el
encabezado `event_time_ms`: el consumidor arma ventanas de 15 s por ese instante, no por cuándo llegan.

In [ ]:
from kafka import KafkaProducer

test_split = json.loads((ROOT / "annotations/reviewed/test.json").read_text())
routes = {}
for image in test_split["images"]:
    video, _, frame = image["file_name"].rpartition("_")
    routes.setdefault(video[:8], []).append((int(frame.split(".")[0]), image["file_name"]))
routes = {r: [name for _, name in sorted(f)] for r, f in sorted(routes.items(), key=lambda kv: -len(kv[1]))[:3]}
{route: len(names) for route, names in routes.items()}

In [ ]:
producer = KafkaProducer(bootstrap_servers="localhost:19092")

def publish(routes, night=False, fps=2, rate=2):
    """Publishes every route frame by frame; event time advances at `fps` per route."""
    start = datetime.now(timezone.utc)
    t0 = (int(time.time() * 1000) // 15_000 + 1) * 15_000          # start at a 15 s window boundary
    for i in range(max(len(names) for names in routes.values())):
        for route, names in routes.items():
            if i >= len(names):
                continue
            frame = Image.open(IMAGES / names[i]).convert("RGB")
            if night:
                frame = ImageEnhance.Brightness(frame).enhance(0.35)
            jpeg = io.BytesIO(); frame.save(jpeg, "JPEG", quality=90)
            producer.send("frames", key=route.encode(), value=jpeg.getvalue(),
                          headers=[("event_time_ms", str(t0 + int(i * 1000 / fps)).encode()),
                                   ("frame_id", f"{route}-{i}".encode())])
            time.sleep(1 / rate)
    producer.flush()
    return start, datetime.now(timezone.utc)

day = publish(routes)

Lo que el consumidor dejó en el Data Lake por cada ventana (lo lee la plataforma, o un tablero):

In [ ]:
s3 = boto3.client("s3", endpoint_url="http://localhost:9000",
                  aws_access_key_id=env["MINIO_ACCESS_KEY"], aws_secret_access_key=env["MINIO_SECRET_ACCESS_KEY"])

def read_json(key):
    return json.loads(s3.get_object(Bucket="data", Key=key)["Body"].read())

def keys_between(prefix, start, end):
    pages = s3.get_paginator("list_objects_v2").paginate(Bucket="data", Prefix=prefix)
    return [o["Key"] for page in pages for o in page.get("Contents", []) if start <= o["LastModified"] <= end]

def windows(stream, wait=25):
    """Window reports written after `stream` started (the consumer closes idle routes after 10 s)."""
    time.sleep(wait)
    start, _ = stream
    keys = keys_between(f"curated/stream_windows/date={start:%Y-%m-%d}/", start, datetime.now(timezone.utc))
    reports = [read_json(key) for key in keys]
    columns = ["camera", "frames", "late_frames", "latency_ms_p50", "latency_ms_p95", "detections_per_frame"]
    return pd.DataFrame(reports)[columns].rename(columns={"camera": "route"}).sort_values("route")

windows(day)

Las particiones del topic y lo que confirmó el grupo de consumidores (*offsets* y retraso):

In [ ]:
print(subprocess.run(["docker", "exec", "redpanda", "rpk", "group", "describe", "detector"],
                     capture_output=True, text=True).stdout)

## 5 · Los mismos recorridos, de noche

Mismos frames, oscurecidos. Las métricas operativas casi no cambian: el problema no es de latencia, es de los datos.

In [ ]:
night = publish(routes, night=True)
windows(night)

## 6 · Drift: ¿cambiaron las condiciones de captura?

En producción no hay etiquetas para medir el mAP: se vigila si las entradas cambian respecto de la referencia (el `val`
revisado, inferido con el mismo modelo). Cada predicción guardada trae el brillo de su imagen; lo comparamos con la
referencia mediante un test de Kolmogorov-Smirnov.

In [ ]:
from scipy.stats import gaussian_kde, ks_2samp

reference = read_json("curated/reference/billboard-detector/v1.json")["samples"]["brightness"]

def brightness(start, end):
    """Brightness of every prediction stored between `start` and `end` (calibrated threshold only)."""
    records = [read_json(key) for key in keys_between(f"predictions/date={start:%Y-%m-%d}/", start, end)]
    return [r["image_stats"]["brightness"] for r in records if r.get("threshold_is_default")]

traffic = {"day": brightness(day[0], night[0]), "night": brightness(night[0], datetime.now(timezone.utc))}
pd.DataFrame([{"route traffic": name, "frames": len(v), "mean brightness": round(sum(v) / len(v), 3),
               "KS p-value": f"{ks_2samp(reference, v).pvalue:.2g}",
               "drift (α = 0.01)": "yes" if ks_2samp(reference, v).pvalue < 0.01 else "no"}
              for name, v in traffic.items()])

In [ ]:
x = [i / 500 for i in range(400)]
fig, ax = plt.subplots(figsize=(11, 3.8))
for label, values, color in [("reference (reviewed val)", reference, "#2a78d6"),
                             ("day routes", traffic["day"], "#1baf7a"),
                             ("night routes", traffic["night"], "#eb6834")]:
    ax.plot(x, gaussian_kde(values)(x), color=color, linewidth=2, label=f"{label} ({len(values)} frames)")
ax.set_xlabel("mean frame brightness (0–1)"); ax.set_yticks([]); ax.legend(frameon=False)
ax.set_title("Night traffic drifts away from the reference; day traffic does not", loc="left");

El recorrido de día (videos que no están en la referencia) no da drift; el de noche sí. Para la plataforma es la señal
de que las etiquetas de ese tráfico pueden ser menos confiables.

El umbral es α = 0,01, no 0,05: el sistema hace cuatro tests por ventana (conteo, score, brillo y contraste) y frames
consecutivos se parecen entre sí, así que con 0,05 habría falsas alarmas. Esto mismo lo hace cada hora el DAG
`drift_monitoring`, sobre todo el tráfico y con los cuatro tests corregidos por Holm: **siguiente paso, en el navegador:
disparar `drift_monitoring` en Airflow y abrir la alerta en MinIO.**